In [1]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Extract and check a promotion signage proof

| | |
|-|-|
| Author(s) | [Matt Robinson](https://github.com/mr394729) |

> **This copy keeps the output of one complete run** (24 September 2026, in a test namespace), so you can read what each cell prints even if a cell fails for you. Your numbers and wording will differ where a model answers. To start clean, choose **Edit > Clear Outputs of All Cells** in JupyterLab, or run `jupyter nbconvert --clear-output --inplace <notebook>`.

## Overview

Before a promotion goes live, each store receives a signage proof: the shelf signs with product, price and dates. Someone has to check every line against the week's promotion plan. This quickstart hands that check to an agent, and keeps the final decision in code.

### Multimodal input and structured output

Gemini reads PDFs and images directly. An `LlmAgent` with an [`output_schema`](https://adk.dev/agents/llm-agents/) returns JSON that matches a Pydantic model, so the next step receives fields, not prose. The uploaded file is kept as an [artifact](https://adk.dev/artifacts/), a versioned file that belongs to the session.

### Generate and review with a loop

A [`LoopAgent`](https://adk.dev/agents/workflow-agents/loop-agents/) runs an extractor and a critic in turn. The critic checks the draft against simple rules and calls `exit_loop` when they hold, so the loop stops itself. (ADK 2.x marks `LoopAgent` as deprecated in favour of a `Workflow` graph; it still works.)

### The model reads, the code decides

A plain Python function compares the extracted lines with the plan. A model upgrade can change how the proof is read, but it cannot change which differences count as errors.

<img width="60%" src="../../docs/diagrams/q07.png" alt="A promotion proof flows through extraction, review and a code comparison with the plan" />

### Objectives

In this tutorial, you will learn how to combine multimodal extraction, a review loop and a deterministic check in one ADK workflow.

You will complete the following tasks:

- Look at the promotion plan and the signage proof
- Read the extraction schema and run the comparison function on your own input
- Run the agent on the proof and read the extracted lines and the discrepancy report

### Costs

This tutorial uses billable components of Google Cloud:

- Gemini on Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing), and use the [Pricing Calculator](https://cloud.google.com/products/calculator/) to generate a cost estimate based on your projected usage.

## Get started

### Set Google Cloud project information

This quickstart reads the store data you loaded in the workshop notebooks, in your own namespace. Set your project ID and the namespace you chose during setup.

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
WORKSHOP_NAMESPACE = "[your-namespace]"  # @param {type: "string"}

if PROJECT_ID == "[your-project-id]":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "")
if WORKSHOP_NAMESPACE == "[your-namespace]":
    WORKSHOP_NAMESPACE = os.environ.get("WORKSHOP_NAMESPACE", "")
if not PROJECT_ID or not WORKSHOP_NAMESPACE:
    raise ValueError("Set PROJECT_ID and WORKSHOP_NAMESPACE above.")

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["WORKSHOP_NAMESPACE"] = WORKSHOP_NAMESPACE
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["STORE_OPS_ENV"] = "dev"

# The quickstart imports shared store code from the repository root, two folders up
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "agents" / "cymbal_store_ops").is_dir())
sys.path.insert(0, str(REPO_ROOT))

### Import libraries

In [3]:
import logging
import warnings

# Keep the output to the agent's own events: ADK marks experimental and deprecated features
# with warnings and logs configuration hints, and the Gen AI SDK logs a note whenever a
# response mixes text and tool calls.
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
logging.getLogger("google_adk").setLevel(logging.ERROR)
logging.getLogger("google_genai").setLevel(logging.ERROR)

In [4]:
import json

import pandas as pd
from agent import PLAN, PromoProof, app, compare
from google.adk.runners import InMemoryRunner
from google.genai import types
from pypdf import PdfReader

## Look at the inputs

### The promotion plan

`promo_plan_2026W40.json` holds the week's approved promotions for store S-014. In production this would be a query on the promotions table.

In [5]:
plan = json.loads(PLAN.read_text())
print(f"Week {plan['week']}")
pd.DataFrame(plan["promotions"])

Week 2026-W40


,product_id,product_name,promo_price_usd,start_date,end_date
0,P-0101,Lumière Hydra Cream,22.99,2026-10-04,2026-10-10
1,P-0420,Noir Velvet Eau de Parfum,79.00,2026-10-04,2026-10-10
2,P-0141,Hydra Moisturizer,9.99,2026-10-04,2026-10-10
3,P-0231,Lift Hair Mask,11.00,2026-10-04,2026-10-10


### The signage proof

`eval/artifacts/promo_proof_S-014_2026W40.pdf` is a one-page proof for the same week. It contains three planted errors. Read its text layer to see what the agent will be given:

In [6]:
PROOF_PDF = Path("eval/artifacts/promo_proof_S-014_2026W40.pdf")

print(PdfReader(PROOF_PDF).pages[0].extract_text())

Cymbal Beauty  ·  Promotion signage proof
Store: S-014 Cymbal Beauty Naperville   ·   Store group: Chicagoland
Promotion week: 2026-W40  (Sunday 2026-10-04 to Saturday 2026-10-10)
Proof version 3  ·  printed 2026-10-01  ·  for the signage team
Line Product ID Product Sign type Promo price Starts Ends
1 P-0101 Lumière Hydra Cream shelf talker $24.99 2026-10-04 2026-10-10
2 P-0420 Noir Velvet Eau de Parfum locked case card $79.00 2026-10-04 2026-10-10
3 P-0141 Hydra Moisturizer end cap sign $9.99 2026-10-04 2026-10-17
4 P-0333 Glow Body Wash shelf talker $25.00 2026-10-04 2026-10-10
5 P-0231 Lift Hair Mask shelf talker $11.00 2026-10-04 2026-10-10
Check every sign against this proof before it goes up (SOP 04). Signs go up before opening on Sunday.


## Look at the agent

### The extraction schema

The extractor must answer with a `PromoProof`: the store, the week and one entry per printed line. The field descriptions tell the model what to copy, and the schema rejects anything that does not fit.

In [7]:
for name, field in PromoProof.model_fields.items():
    print(f"{name:20} {field.description or ''}")

store_id             like S-014
week                 promotion week like 2026-W40
lines                
confidence           0-1; below 0.7 goes to a person
needs_human_review   


### The comparison

`compare()` is ordinary Python. Give it a proof with one wrong price and it reports exactly that line, with no model involved:

In [8]:
first = plan["promotions"][0]
test_proof = {
    "store_id": "S-014",
    "week": plan["week"],
    "lines": [
        {
            "line": 1,
            "product_id": first["product_id"],
            "product_name": first["product_name"],
            "sign_type": "shelf talker",
            "promo_price_usd": first["promo_price_usd"] + 2,
            "start_date": first["start_date"],
            "end_date": first["end_date"],
        }
    ],
}

# a missing sign has no line on the proof: an integer column shows it as <NA> rather than NaN
pd.DataFrame(compare(test_proof, plan)).astype({"line": "Int64"})

,line,product_id,product_name,issue,proof,plan
0,1,P-0101,Lumière Hydra Cream,price_mismatch,$24.99,$22.99
1,<NA>,P-0420,Noir Velvet Eau de Parfum,missing_sign,no sign,$79.00
2,<NA>,P-0141,Hydra Moisturizer,missing_sign,no sign,$9.99
3,<NA>,P-0231,Lift Hair Mask,missing_sign,no sign,$11.00


The single wrong price is reported, and every other planned product is reported as a missing sign, because this test proof only has one line.

### The workflow

The app runs the extractor and critic in a loop of at most two rounds, then a checker agent that calls `compare_promo_proof` and writes the report:

In [9]:
def show_tree(agent, depth=0):
    print("  " * depth + f"{type(agent).__name__}: {agent.name}")
    for sub_agent in getattr(agent, "sub_agents", []):
        show_tree(sub_agent, depth + 1)


show_tree(app.root_agent)

SequentialAgent: document_extraction_agent
  LoopAgent: extract_and_review
    LlmAgent: extractor
    LlmAgent: critic
  LlmAgent: checker


## Run the agent

Create a runner for the app and a session. The app includes `SaveFilesAsArtifactsPlugin`, which stores any file attached to a message as an artifact before the agents run. A `before_model_callback` on the extractor, `attach_proof`, loads that artifact and adds it to every extraction request, so each round of the loop reads the same file.

In [10]:
runner = InMemoryRunner(app=app)
session = await runner.session_service.create_session(app_name=app.name, user_id="dana")

Send the proof as a PDF part next to the request. `display_name` becomes the artifact's file name. The cell prints each tool call and the checker's report; the run takes about 20 seconds.

In [11]:
message = types.Content(
    role="user",
    parts=[
        types.Part(
            inline_data=types.Blob(
                mime_type="application/pdf",
                data=PROOF_PDF.read_bytes(),
                display_name=PROOF_PDF.name,
            )
        ),
        types.Part(text="Check the promotion signage proof I attached against this week's plan."),
    ],
)

async for event in runner.run_async(
    user_id=session.user_id, session_id=session.id, new_message=message
):
    for call in event.get_function_calls():
        print(f"[{event.author}] calls {call.name}")
    if event.author == "checker" and event.is_final_response() and event.content:
        print("\n" + "".join(part.text or "" for part in event.content.parts if not part.thought))

[critic] calls exit_loop


[checker] calls compare_promo_proof



There are 3 discrepancies:

- Line 1: P-0101 (Lumière Hydra Cream) - Proof says $24.99, Plan says $22.99
- Line 3: P-0141 (Hydra Moisturizer) - Proof says 2026-10-17, Plan says 2026-10-10
- Line 4: P-0333 (Glow Body Wash) - Proof says $25.00, Plan says no promotion this week


The extractor prints no tool call because it answers with its schema. The critic called `exit_loop` in the first round, so the loop ran once. The report should name three discrepancies: P-0101 printed at $24.99 (plan $22.99), P-0141 ending on 2026-10-17 (plan 2026-10-10), and P-0333, which is not on promotion this week. The wording differs from run to run; the three lines should match, because `compare()` finds them.

### Read the extracted proof

The extractor wrote its result to the session state under `proof` (its `output_key`). This is what the comparison ran on:

In [12]:
session = await runner.session_service.get_session(
    app_name=app.name, user_id=session.user_id, session_id=session.id
)
proof = session.state["proof"]
print(f"Store {proof['store_id']}, week {proof['week']}, confidence {proof['confidence']}")
pd.DataFrame(proof["lines"])

Store S-014, week 2026-W40, confidence 0.99


,line,product_id,product_name,sign_type,promo_price_usd,start_date,end_date
0,1,P-0101,Lumière Hydra Cream,shelf talker,24.99,2026-10-04,2026-10-10
1,2,P-0420,Noir Velvet Eau de Parfum,locked case card,79.00,2026-10-04,2026-10-10
2,3,P-0141,Hydra Moisturizer,end cap sign,9.99,2026-10-04,2026-10-17
3,4,P-0333,Glow Body Wash,shelf talker,25.00,2026-10-04,2026-10-10
4,5,P-0231,Lift Hair Mask,shelf talker,11.00,2026-10-04,2026-10-10


All five printed lines come back as fields, including P-0333, which is not in the plan. The extractor copied the proof as printed; deciding that line 4 is an error was the comparison's job. A confidence below 0.7 would set the proof aside for a person.

### Try it yourself

Change `test_proof` above so its price matches the plan and run the comparison again, or edit the plan JSON and rerun the agent to see the report change without touching any prompt.

## Cleaning up

This notebook creates no cloud resources. The session and the artifact lived in memory and end when you restart the kernel.

## What's next

- [Loop agents](https://adk.dev/agents/workflow-agents/loop-agents/)
- [Artifacts](https://adk.dev/artifacts/)
- [Quickstart guide](README.md) for running this agent in the ADK developer UI
- Next quickstart: [Tools over MCP](../08-mcp-tools-agent/walkthrough.ipynb)